In [2]:
import os
import json
import random
from openai import OpenAI
import pandas as pd
from tqdm import tqdm


# Get prediction answer

In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Paths
base_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/pororo"
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")

# Load dataset
with open(qa_json_path, "r") as f:
    qa_data = json.load(f)["PororoQA"]

# Load descriptions
descriptions = pd.read_csv(description_csv_path)

def get_prediction(question, gif_paths, description, subtitles):
    images = []
    for gif_path in gif_paths:
        with open(gif_path, "rb") as gif_file:
            images.append(gif_file.read())

    prompt = f"""
    Question: {question}
    Description: {description}
    Subtitles: {subtitles}
    Images: {len(images)} GIF frames

    Provide an answer based on the description, subtitles, and images.
    """

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150,
        temperature=0.7,
    )
    return completion.choices[0].message.content.strip()

# Check predicted answer

In [4]:
# def check_answer(correct_answer, predicted_answer):
#     prompt = f"""
#     Correct Answer: {correct_answer}
#     Predicted Answer: {predicted_answer}

#     Is the predicted answer correct? Answer with "1" for correct and "0" for incorrect.
#     """

#     completion = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}],
#         max_tokens=10,
#         temperature=0.3,
#     )
#     return int(completion.choices[0].message.content.strip())

def check_answer(correct_answer, predicted_answer):
    """
    Compare the correct answer with the predicted answer using semantic similarity
    Returns: float between 0 and 1 indicating the similarity/correctness
    """
    prompt = f"""
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Rate how well the predicted answer matches the correct answer on a scale of 0 to 1:
    - 1.0: Perfect match or completely correct meaning
    - 0.75: Mostly correct with minor differences
    - 0.5: Partially correct
    - 0.25: Slightly correct but missing key points
    - 0.0: Completely incorrect or unrelated

    Provide only the numeric score (e.g. 0.75) with no other text.
    """

    completion = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0.3  # Lower temperature for more consistent scoring
    )
    
    try:
        score = float(completion.choices[0].message.content.strip())
        # Ensure score is between 0 and 1
        return max(0.0, min(1.0, score))
    except ValueError:
        # Return 0 if we can't parse the score
        return 0.0


# Processe data and calculate accuracy

In [5]:
def get_seeded_question(questions, gif_num, base_seed=42):
    """
    Get a deterministic random question for a GIF using fixed seed
    Args:
        questions: List of questions for current GIF
        gif_num: Current GIF number
        base_seed: Base random seed
    Returns:
        Selected question entry
    """
    if not questions:
        return None
    # Create new Random instance for each GIF
    local_random = random.Random(base_seed + gif_num)
    # Sort questions to ensure consistent ordering
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

# Set random seed for reproducibility
random.seed(42)

# Create list to store evaluation results
evaluation_results = []

# TODO 修改，增加别的video
# First, filter questions only for episode 1
ep1_questions = [q for q in qa_data if "Pororo_ENGLISH1_1_ep1" in q["video_name"]]

# Group questions by supporting_num
grouped_questions = {}
for entry in ep1_questions:
    supporting_num = entry["supporting_num"]
    if supporting_num not in grouped_questions:
        grouped_questions[supporting_num] = []
    grouped_questions[supporting_num].append(entry)

# Process GIFs 1-36 in sequence
gif_numbers = list(range(1, 37))
correct_count = 0
total_count = len(gif_numbers)
gif_accuracy = {}

# Process each GIF number in sequence
for gif_num in tqdm(gif_numbers, total=total_count):
    current_gif_questions = grouped_questions.get(str(gif_num), [])
    
    if not current_gif_questions:
        print(f"No questions found for GIF {gif_num}")
        continue
        
    # Use seeded random selection
    entry = get_seeded_question(current_gif_questions, gif_num)
    
    # Get question info
    video_name = entry["video_name"]
    question = entry["question"]
    correct_idx = entry["correct_idx"]
    answers = [entry[f"answer{i}"] for i in range(5)]
    correct_answer = answers[correct_idx]
    qid = entry["qid"]

    # Construct paths
    episode_folder = os.path.join(base_dir, "Scenes_Dialogues", "Pororo_ENGLISH1_1", "Pororo_ENGLISH1_1_ep1")
    subtitles_path = os.path.join(episode_folder, "subtitles.txt")
    
    # Load subtitles
    with open(subtitles_path, "r") as f:
        subtitles = f.read()

    # Process current GIF
    gif_files = [f"{gif_num}.gif"]
    gif_paths = [os.path.join(episode_folder, gif_file) for gif_file in gif_files]

    # Get description
    description_row = descriptions.loc[descriptions.iloc[:, 0] == video_name]
    if description_row.empty:
        print(f"Description for {video_name} not found")
        continue
    description = description_row.iloc[0, 2]

    # Get prediction
    predicted_answer = get_prediction(question, gif_paths, description, subtitles)

    # Calculate accuracy
    is_correct = predicted_answer is not None and check_answer(correct_answer, predicted_answer)
    correct_count += is_correct

    # Store current result
    result = {
        'gif_num': gif_num,
        'video_name': video_name,
        'qid': qid,
        'question': question,
        'correct_answer': correct_answer,
        'predicted_answer': predicted_answer,
        'accuracy': (is_correct * 100)  # Convert to percentage
    }
    evaluation_results.append(result)

    # Print debugging info
    print(f"Video name: {video_name}")
    print(f"GIF number: {gif_num}")
    print(f"QID: {qid}")
    print(f"Question: {question}")
    print(f"Correct Answer: {correct_answer}")
    print(f"Predicted Answer: {predicted_answer}")
    print(f"GIF number: {gif_num}, Accuracy: {(is_correct * 100):.0f}%")

# Calculate overall accuracy
average_accuracy = correct_count / total_count
print(f"Average Accuracy: {average_accuracy * 100:.2f}%")

  3%|▎         | 1/36 [00:03<02:13,  3.81s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 1
QID: 4
Question: what is the nature of place where pororo leaves?
Correct Answer: it is the cold country , small village
Predicted Answer: Pororo lives in a cold, snowy environment characterized by a small village nestled in a forest on the mountains. This setting is filled with trees that are covered in snow, emphasizing the wintry climate of the place. The village is likely quaint and charming, providing a backdrop for Pororo's adventures. The presence of a baby dinosaur hatching from an egg adds an element of surprise and excitement to this otherwise serene and snowy landscape. Overall, it's a whimsical and enchanting place that fosters curiosity and friendship, as seen in Pororo's interactions with his new dinosaur friend, Crong.
GIF number: 1, Accuracy: 75%


  6%|▌         | 2/36 [00:06<01:43,  3.03s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 2
QID: 83
Question: what is the name of the little penguin
Correct Answer: the name of the little penguin is pororo
Predicted Answer: The name of the little penguin is Pororo.
GIF number: 2, Accuracy: 100%


  8%|▊         | 3/36 [00:07<01:12,  2.20s/it]

...

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 34
QID: 655
Question: what phrase do all of the animals say to the green dinosaur crung?
Correct Answer: they all say, "nice to meet you".
Predicted Answer: In the story, all of the animals say "crong" to the green dinosaur Crong.
GIF number: 34, Accuracy: 0%


 97%|█████████▋| 35/36 [01:26<00:01,  1.90s/it]

Video name: Pororo_ENGLISH1_1_ep10
GIF number: 35
QID: 1119
Question: was pororo enjoying the new ride?
Correct Answer: yes, pororo was enjoying and said " yahoo"
Predicted Answer: Based on the information provided, it doesn't explicitly mention whether Pororo was enjoying a new ride. The description and subtitles focus more on Pororo's interactions with the baby dinosaur, Crong, and their new friendship. There is no indication of a ride or Pororo's feelings towards it in the provided text. Therefore, we cannot conclude that Pororo was enjoying a new ride.
GIF number: 35, Accuracy: 0%


100%|██████████| 36/36 [01:27<00:00,  2.44s/it]

Video name: Pororo_ENGLISH1_1_ep1
GIF number: 36
QID: 672
Question: who did pororo become friends with today?
Correct Answer: he became friends with baby dinosaur, crong
Predicted Answer: Today, Pororo became friends with a baby dinosaur named Crong.
GIF number: 36, Accuracy: 100%
Average Accuracy: 39.58%


# Save data

In [14]:
# Remove any existing Average rows
evaluation_results = [r for r in evaluation_results if r['gif_num'] != 'Average Accuracy']

# Add average accuracy row using the average_accuracy from main processing
average_result = {
    'gif_num': 'Average Accuracy',  
    'video_name': '',
    'qid': '',
    'question': '',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy * 100  
}
evaluation_results.append(average_result)

# Save to CSV 
output_path = os.path.join("/Users/wt/PythonProjects/MultimodalComicAgent/results", 
                          'pororo_evaluation_results_single_agent.csv')

# Save as DataFrame and force overwrite
results_df = pd.DataFrame(evaluation_results)
results_df.to_csv(output_path, index=False, mode='w')
print(f"Results saved to: {output_path}")
print(f"Total questions: {len(evaluation_results)-1}")  
print(f"Average accuracy: {average_accuracy*100:.2f}%")

Results saved to: /Users/wt/PythonProjects/MultimodalComicAgent/results/pororo_evaluation_results_single_agent.csv
Total questions: 36
Average accuracy: 39.58%
